En este archivo Notebook realizo diferentes implementaciones del proyecto para poder ejecutarlas por bloques y obtener un resultado visual

In [1]:
# Añadimos todas las librerias necesarias
import oracledb as oracledb
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

Creación de la conexión a la base de datos

In [13]:
# Crear el DSN usando la IP remota
dsn = oracledb.makedsn("afrodita.lcc.uma.es", 1521, sid="APOLO")

# Conectarse
conn = oracledb.connect(user="tfm_puertas", password="JCGRmlbEsc", dsn=dsn)

# Ejecutar consulta
cur = conn.cursor()
cur.execute("select distinct nombreasignatura from v_calificaciones")
for row in cur:
    print(row)

cur.close()

('Bases de Datos',)
('Estructura de Datos',)
('Ingeniería y Ciencia de Datos II',)
('Industrialización y Despliegue de Sistemas loT',)
('Planificación de Proyectos y Análisis de Riesgos',)
('Control Automático',)
('Programación de Videojuegos',)
('Diseño de Sistemas Operativos',)
('Bioquímica Estructural',)
('Física II',)
('Laboratorio de Computación Científica',)
('Administración de Bases de Datos',)
('Seguridad en Sistemas Industriales y Ciberfísicos',)
('Visión por Computador',)
('Sistemas de Información Empresarial',)
('Arquitecturas Paralelas',)
('Programación Distribuida',)
('Ingeniería del Software Avanzada',)
('Álgebra Lineal y Geometría',)
('Análisis Matemático III',)
('Telemedicina',)
('Inferencia Estadística',)
('Geometría Diferencial Global de Superficies',)
('Teoría de Autómatas y Lenguajes Formales',)
('Desarrollo de Aplicaciones en la Nube',)
('Sistemas de Información para Internet',)
('Diseño y Evaluación de Infraestructuras Informáticas',)
('Cognición y Comunicación en

Función que calcula la nota media ponderada.

Nota media ponderada: Media de todas las asignaturas aprobadas por el número de créditos que valen, dividido entre el total de créditos de la titulación

In [4]:
def calcular_media_ponderada(conn, codigo_alumno, curso_max):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT 
                AVG(TO_NUMBER(NUM_CALIFICACIÓN)) AS media_aprobadas,
                COUNT(*) AS asignaturas_aprobadas
            FROM v_calificaciones
            WHERE CODIGOALUM = :codigo
            AND CALIFICACIÓN NOT IN ('NO PRESENTADO', 'SUSPENSO')
            AND TO_NUMBER(SUBSTR(CURSOACADÉMICO, 1, 4)) < TO_NUMBER(SUBSTR(:curso_inicio , 1, 4))
        """, codigo=codigo_alumno, curso_inicio=curso_max[:4])

        resultado = cur.fetchone() #Como esperamos solo un resultado, uso fetchone
        media, aprobadas = resultado

        if media is None or aprobadas == 0:
            return None  # Evitamos división por cero o falta de datos

        # Fórmula: (media * aprobadas * 6) / 240. NOTA: Supongo que todas las asignaturas valen 6 créditos ya que no tenemos información de cada una
        ponderada = (media * aprobadas * 6) / 240
        return round(ponderada, 2)
    
calcular_media_ponderada(conn, '0208F18506E41D3F29A4CAAD842FD0FA','2020-21')


1.57

Dada una asignatura, obtener todos los alumnos que la han cursado, y el curso académico en el que lo hicieron

In [5]:
def obtener_alumnos_matriculados(conn, nombre_asignatura):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT DISTINCT CODIGOALUM, CURSOACADÉMICO
            FROM v_calificaciones
            WHERE NOMBREASIGNATURA = :asignatura
        """, asignatura=nombre_asignatura)

        return cur.fetchall()
        
obtener_alumnos_matriculados(conn, 'Visión por Computador')


[('E7AC1E0B6F38D865A51141CEAA6AABDC', '2018-19'),
 ('0154E5C78A6179BDB090C06DEC095A17', '2018-19'),
 ('7334688E1698AECCCB5C2693A55E91DB', '2019-20'),
 ('09D3DA93C1195F9337C98092057E86BE', '2019-20'),
 ('56D2035062EF09C99B875AF14B6DBAEF', '2019-20'),
 ('F2D77DDA93F07980CDAFA4FC9B49B81F', '2020-21'),
 ('E475FCF6F6FB2F4200647487CBBA367B', '2020-21'),
 ('E8C0BD34336D34FB3D50886305A9DE12', '2020-21'),
 ('592AEBD3C172B1FF1B0017D1296ED7E8', '2021-22'),
 ('9475ED7552FA67FD81C807CACB4497F4', '2021-22'),
 ('B532D6CA33EAE40DB254D225231E4986', '2022-23'),
 ('B4A95BD04DD3E641CBDF84E3F3416C64', '2022-23'),
 ('8FAB2C4EDCA1484446405F73B2EC2026', '2022-23'),
 ('7114F27ECC34C1E75F8557E099343544', '2020-21'),
 ('6D4F20745BAB50857886AAD4AFD339D5', '2021-22'),
 ('47BA4947D4B7CE8520F27723ABC03B29', '2020-21'),
 ('43ECD972302EE21A6B0ECCBB67940670', '2018-19'),
 ('D7F78EB38BBCCC219CC12770E0CCD213', '2018-19'),
 ('8E14F1B5DE2DEEC5FED6D376A2A53E72', '2019-20'),
 ('E08B5D4B37BF613FDA0E9D477207702C', '2019-20'),


Calcular nota de corte de cada año

Nota de corte: La nota media ponderada más baja que está matriculado en la asignatura hasta el año anterior de la matrícula

In [6]:
def calcular_nota_corte(conn, nombre_asignatura):
    # 1. Obtener alumnos + curso académico donde cursaron esa asignatura
    alumnos_con_curso = obtener_alumnos_matriculados(conn, nombre_asignatura)

    # 2. Agrupar alumnos por curso académico
    cursos = {}
    for codigo_alum, curso_acad in alumnos_con_curso:
        cursos.setdefault(curso_acad, []).append(codigo_alum)

    # 3. Para cada curso, calcular la nota de corte
    nota_corte_por_anio = {}

    for curso_acad, codigos_alumnos in cursos.items():
        medias = []
        for codigo_alum in codigos_alumnos:
            media = calcular_media_ponderada(conn, codigo_alum, curso_acad)
            if media is not None:
                #print(media, ',', curso_acad) #Para ver las medias ponderadas con el año al que pertenecen
                medias.append(media)
        
        if medias:
            nota_corte_por_anio[curso_acad] = min(medias)

    return nota_corte_por_anio  # Dict: { "2018-19": 6.25, "2019-20": 5.8, ... }

calcular_nota_corte(conn, 'Visión por Computador')


{'2019-20': 0.17, '2020-21': 0.28, '2021-22': 0.3, '2022-23': 2.12}

Cálculo de la probabilidad de entrada: Dada la nota de corte de cada año, el porcentaje de años en los que habría entrado en la asignatura

In [44]:
def calcular_probabilidad_entrada(conn, nombre_asignatura, codigo_alumno):
    # Obtener las notas de corte por año
    notas_corte = calcular_nota_corte(conn, nombre_asignatura)
    print(notas_corte)

    if not notas_corte:
        return 0.0  # No hay base para calcular probabilidad

    total = 0
    supera = 0

    media_alumno = calcular_media_ponderada(conn, codigo_alumno, '2022-23')
    print(media_alumno)
    #media_alumno = 0.15
    for curso_acad, nota_corte in notas_corte.items():
        
        if media_alumno is not None:
            total += 1
            if media_alumno > nota_corte:
                supera += 1

    if total == 0:
        return 0.0  # El alumno no tiene historial válido

    probabilidad = (supera / total) * 100
    return round(probabilidad, 2)

calcular_probabilidad_entrada(conn, 'Visión por Computador', '0208F18506E41D3F29A4CAAD842FD0FA')


{'2019-20': 0.17, '2020-21': 0.28, '2021-22': 0.3, '2022-23': 2.12}
3.33


100.0

In [ ]:
# Suponemos que tienes un DataFrame con columnas: ['CODIGOALUM', 'NOMBREASIGNATURA', 'NUM_CALIFICACIÓN']
df = pd.read_csv("data.csv")
def entrenar_clustering(df, n_clusters=10):
    # Crear matriz alumno-asignatura
    matriz = df.pivot_table(index='CODIGOALUM', columns='NOMBREASIGNATURA', values='NUM_CALIFICACIÓN')
    matriz = matriz.fillna(0)  #Para evitar valores NaN
    
    scaler = StandardScaler()
    matriz_escalada = scaler.fit_transform(matriz)
    
    modelo = KMeans(n_clusters=n_clusters, random_state=42)
    modelo.fit(matriz_escalada)
    
    # Añadir etiquetas de cluster
    df_clusters = pd.DataFrame(matriz.index, columns=['CODIGOALUM'])
    df_clusters['cluster'] = modelo.labels_
    
    return modelo, scaler, matriz, df_clusters


969140


In [14]:
def predecir_afinidad_cluster(alumno_id, asignatura, modelo, scaler, matriz, df_clusters, df_original):
    if alumno_id not in matriz.index or asignatura not in matriz.columns:
        return None
    
    alumno_vector = scaler.transform(matriz.loc[[alumno_id]])
    cluster_id = modelo.predict(alumno_vector)[0]
    
    # Alumnos en ese cluster
    alumnos_similares = df_clusters[df_clusters['cluster'] == cluster_id]['CODIGOALUM']
    
    # Notas en la asignatura de ese grupo
    notas = df_original[
        (df_original['CODIGOALUM'].isin(alumnos_similares)) &
        (df_original['NOMBREASIGNATURA'] == asignatura) &
        (df_original['NUM_CALIFICACIÓN'].notnull())
    ]['NUM_CALIFICACIÓN'].astype(float)
    
    if notas.empty:
        return 0.0
    
    return round(notas.mean() / 10, 3)


In [17]:
modelo, scaler, matriz, df_clusters = entrenar_clustering(df)
afinidad = predecir_afinidad_cluster('0208F18506E41D3F29A4CAAD842FD0FA', 'Visión por Computador', modelo, scaler, matriz, df_clusters, df)
print(f"Afinidad estimada: {afinidad}")

Afinidad estimada: 0.442


In [8]:
#Cierre de conexión
conn.close()